In [5]:
# ============================================================
# CLASIFICACIÓN DE HOJAS — KNN v3
# Dataset: 114 muestras | 7 grupos (C, X, J, D, V, SG, S)
# Clases: 0=Alargada, 1=Ovalada, 2=Estrellada, 3=Helecho
# ============================================================
# Cambios respecto a v2:
#  - Dataset actualizado: n=114 (antes n=75)
#  - Grupos nuevos: J (12), D (8), V (6), SG (7), S (6)
#  - Distribución: Alargada=54, Ovalada=23, Estrellada=30, Helecho=7
#  - Misma arquitectura: LOO-CV + GridSearchCV + permutation importance
# ============================================================

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations

from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, label_binarize
from sklearn.model_selection import (
    train_test_split, LeaveOneOut, StratifiedKFold,
    GridSearchCV, cross_val_score
)
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    roc_curve, auc, precision_recall_curve, average_precision_score,
    f1_score
)
from sklearn.inspection import permutation_importance

# ── Estilo global ──────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "DejaVu Sans",
    "axes.spines.top":    False,
    "axes.spines.right":   False,
    "axes.grid":          True,
    "grid.alpha":         0.25,
    "figure.dpi":         130,
})

NOMBRES_CLASES = ["Alargada", "Ovalada", "Estrellada", "Helecho"]
PALETA         = ["#2176AE", "#F7934C", "#2DC653", "#C1121F"]
MARCADORES     = ["o", "s", "^", "D"]

# ── Ruta de salida ─────────────────────────────────────────
# Cambia OUTPUT_DIR a "." si ejecutas en local
OUTPUT_DIR = "."   # <- ajusta aquí si es necesario

# ============================================================
# 1. CARGA Y FEATURE ENGINEERING
# ============================================================
CSV_PATH = r"C:\Users\santi\Downloads\hojas_combinadas.csv"

df = pd.read_csv(CSV_PATH, sep=None, engine='python')
df.columns = df.columns.str.strip().str.lower()

# Renombrar columnas al esquema interno
df = df.rename(columns={
    "largo":      "Largo",
    "ancho":      "Ancho",
    "n_vertices": "Vertices",
    "clase":      "Clase",
})

# Features derivadas
df["ratio"]    = df["Largo"] / df["Ancho"]           # elongación
df["area_est"] = df["Largo"] * df["Ancho"]           # área rectángulo envolvente
df["log_vert"] = np.log1p(df["Vertices"])            # log(Vertices+1)

FEATURES = ["log_vert", "ratio", "area_est"]

X_all = df[FEATURES].values
y_all = df["Clase"].values

# ── Extraer grupo de cada ID ────────────────────────────────
df["grupo"] = df["id"].str.extract(r"([A-Za-z]+)")

print("=" * 65)
print("DATASET — n=114 muestras | 7 grupos")
print("=" * 65)
print("\nDistribución por grupo:")
for g, cnt in df["grupo"].value_counts().sort_index().items():
    print(f"  {g:4s}: {cnt} muestras")

print("\nDistribución por clase:")
for i, nombre in enumerate(NOMBRES_CLASES):
    cnt = int((y_all == i).sum())
    barra = "█" * cnt
    print(f"  {i} {nombre:12s}: {barra} ({cnt})")
print(f"\n  Total: {len(df)} muestras")

# ============================================================
# 2. PARTICIÓN ESTRATIFICADA  60 / 20 / 20
# ============================================================
try:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.20, random_state=42, stratify=y_all)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr,  y_tr,  test_size=0.25, random_state=42, stratify=y_tr)
except ValueError:
    X_tr, X_te, y_tr, y_te = train_test_split(
        X_all, y_all, test_size=0.20, random_state=42)
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_tr,  y_tr,  test_size=0.25, random_state=42)

scaler  = StandardScaler().fit(X_tr)
X_tr_s  = scaler.transform(X_tr)
X_va_s  = scaler.transform(X_va)
X_te_s  = scaler.transform(X_te)

print(f"\n  Entrenamiento: {len(X_tr)} | Validación: {len(X_va)} | Prueba: {len(X_te)}")

# ============================================================
# 3. SELECCIÓN DE K — LEAVE-ONE-OUT CV
# ============================================================
loo     = LeaveOneOut()
k_range = range(1, min(15, len(X_tr)))
loo_scores = []

for k in k_range:
    knn_loo = KNeighborsClassifier(
        n_neighbors=k, metric="euclidean", weights="distance")
    scores = cross_val_score(knn_loo, X_tr_s, y_tr, cv=loo, scoring="accuracy")
    loo_scores.append(scores.mean())

k_optimo_loo = list(k_range)[np.argmax(loo_scores)]
print(f"\n  K óptimo LOO-CV : {k_optimo_loo}  (acc = {max(loo_scores):.3f})")

# ── GridSearchCV ────────────────────────────────────────────
param_grid = {
    "n_neighbors": list(k_range),
    "metric":      ["euclidean", "manhattan", "minkowski"],
    "weights":     ["uniform", "distance"],
}
n_splits = min(5, int(min(np.bincount(y_tr))))
cv_inner = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    KNeighborsClassifier(), param_grid,
    cv=cv_inner, scoring="f1_macro", refit=True, n_jobs=-1
)
grid_search.fit(X_tr_s, y_tr)
k_gs    = grid_search.best_params_["n_neighbors"]
dist_gs = grid_search.best_params_["metric"]
w_gs    = grid_search.best_params_["weights"]
print(f"  GridSearch best : k={k_gs}, metric={dist_gs}, "
      f"weights={w_gs}, F1-macro={grid_search.best_score_:.3f}")

# ── Figura 1: selección de k ────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(list(k_range), loo_scores, marker="o", color=PALETA[0],
        linewidth=2, markersize=7, label="Accuracy LOO-CV")
ax.axvline(k_optimo_loo, color=PALETA[3], linestyle="--", linewidth=1.8,
           label=f"k óptimo LOO = {k_optimo_loo}")
ax.axvline(k_gs, color=PALETA[2], linestyle=":", linewidth=1.8,
           label=f"k GridSearch = {k_gs}")
ax.set_xlabel("Número de vecinos (k)", fontsize=12)
ax.set_ylabel("Accuracy (LOO-CV)", fontsize=12)
ax.set_title("Selección de k — Leave-One-Out sobre entrenamiento\n"
             f"Dataset actualizado: n={len(df)} muestras", fontsize=12, pad=10)
ax.set_xticks(list(k_range))
ax.set_ylim(0, 1.05)
ax.legend(fontsize=10)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_01_seleccion_k.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_01_seleccion_k.png")

# ============================================================
# 4. PREDICCIONES CON MEJOR MODELO (GridSearch)
# ============================================================
knn       = grid_search.best_estimator_
y_pred_va = knn.predict(X_va_s)
y_pred_te = knn.predict(X_te_s)
acc_va    = accuracy_score(y_va, y_pred_va)
acc_te    = accuracy_score(y_te, y_pred_te)

# ============================================================
# 5. MATRICES DE CONFUSIÓN
# ============================================================
def plot_cm(y_true, y_pred, titulo, ax):
    clases    = sorted(set(y_true) | set(y_pred))
    etiquetas = [NOMBRES_CLASES[c] for c in clases]
    cm        = confusion_matrix(y_true, y_pred, labels=clases)
    cm_norm   = cm.astype(float) / (cm.sum(axis=1, keepdims=True) + 1e-9)
    sns.heatmap(cm_norm, annot=cm, fmt="d", cmap="Blues",
                xticklabels=etiquetas, yticklabels=etiquetas,
                ax=ax, cbar=False, linewidths=0.8, linecolor="#ddd",
                vmin=0, vmax=1)
    for i in range(len(cm)):
        ax.add_patch(plt.Rectangle((i, i), 1, 1, fill=False,
                                   edgecolor="#2DC653", lw=2.5))
    ax.set_title(f"{titulo}\nAccuracy: {accuracy_score(y_true, y_pred):.2f}",
                 fontsize=11, pad=8)
    ax.set_xlabel("Predicho", fontsize=10)
    ax.set_ylabel("Real",     fontsize=10)
    ax.tick_params(axis="x", rotation=30, labelsize=9)
    ax.tick_params(axis="y", rotation=0,  labelsize=9)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_cm(y_va, y_pred_va, "Validación", axes[0])
plot_cm(y_te, y_pred_te, "Prueba",     axes[1])
fig.suptitle(
    f"KNN — Matrices de confusión  (k={k_gs}, {dist_gs}, weights={w_gs})\n"
    f"Dataset: n={len(df)} muestras",
    fontsize=12, y=1.01
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_02_matrices_confusion.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_02_matrices_confusion.png")

# ============================================================
# 6. REPORTES DE MÉTRICAS
# ============================================================
def evaluar(y_true, y_pred, nombre):
    print(f"\n{'─'*60}")
    print(f"  {nombre.upper()}  (n={len(y_true)})")
    print(f"{'─'*60}")
    clases   = sorted(set(y_true) | set(y_pred))
    nombres  = [NOMBRES_CLASES[c] for c in clases]
    ausentes = [NOMBRES_CLASES[c] for c in range(4) if c not in clases]
    print(classification_report(y_true, y_pred, labels=clases,
                                 target_names=nombres, zero_division=0))
    if ausentes:
        print(f"  Sin ejemplos: {', '.join(ausentes)}")

evaluar(y_va, y_pred_va, "Validación")
evaluar(y_te, y_pred_te, "Prueba")

# ============================================================
# 7. CURVAS ROC
# ============================================================
classes_all = [0, 1, 2, 3]
y_score_va  = knn.predict_proba(X_va_s)
y_score_te  = knn.predict_proba(X_te_s)
y_va_bin    = label_binarize(y_va, classes=classes_all)
y_te_bin    = label_binarize(y_te, classes=classes_all)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_bin, y_score, titulo in zip(
        axes,
        [y_va_bin, y_te_bin],
        [y_score_va, y_score_te],
        ["Validación", "Prueba"]):
    for i, (color, nombre) in enumerate(zip(PALETA, NOMBRES_CLASES)):
        if y_bin[:, i].sum() == 0:
            continue
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_score[:, i])
        roc_auc     = auc(fpr, tpr)
        ax.plot(fpr, tpr, color=color, linewidth=2,
                label=f"{nombre}  (AUC={roc_auc:.2f})")
    ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5)
    ax.set_xlabel("Tasa de Falsos Positivos", fontsize=11)
    ax.set_ylabel("Tasa de Verdaderos Positivos", fontsize=11)
    ax.set_title(f"Curvas ROC — {titulo}", fontsize=12)
    ax.legend(fontsize=9, loc="lower right")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_03_roc_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_03_roc_curves.png")

# ============================================================
# 8. CURVAS PRECISIÓN-RECALL
# ============================================================
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, y_bin, y_score, titulo in zip(
        axes,
        [y_va_bin, y_te_bin],
        [y_score_va, y_score_te],
        ["Validación", "Prueba"]):
    for i, (color, nombre) in enumerate(zip(PALETA, NOMBRES_CLASES)):
        if y_bin[:, i].sum() == 0:
            continue
        prec, rec, _ = precision_recall_curve(y_bin[:, i], y_score[:, i])
        ap           = average_precision_score(y_bin[:, i], y_score[:, i])
        ax.plot(rec, prec, color=color, linewidth=2,
                label=f"{nombre}  (AP={ap:.2f})")
    ax.set_xlabel("Recall",    fontsize=11)
    ax.set_ylabel("Precisión", fontsize=11)
    ax.set_title(f"Curvas Precisión-Recall — {titulo}", fontsize=12)
    ax.legend(fontsize=9, loc="upper right")
    ax.set_xlim(-0.02, 1.02)
    ax.set_ylim(-0.02, 1.05)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_04_pr_curves.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_04_pr_curves.png")

# ============================================================
# 9. FRONTERAS DE DECISIÓN — TODOS LOS PARES
# ============================================================
pares = list(combinations(range(len(FEATURES)), 2))
fig, axes = plt.subplots(1, len(pares), figsize=(6 * len(pares), 5))
if len(pares) == 1:
    axes = [axes]

h = 0.03
for ax, (i, j) in zip(axes, pares):
    sc2   = StandardScaler()
    X_tr2 = sc2.fit_transform(X_tr[:, [i, j]])
    X_te2 = sc2.transform(X_te[:, [i, j]])
    knn2  = KNeighborsClassifier(n_neighbors=k_gs, metric=dist_gs, weights=w_gs)
    knn2.fit(X_tr2, y_tr)

    xmin, xmax = X_tr2[:, 0].min() - 0.8, X_tr2[:, 0].max() + 0.8
    ymin, ymax = X_tr2[:, 1].min() - 0.8, X_tr2[:, 1].max() + 0.8
    xx, yy = np.meshgrid(np.arange(xmin, xmax, h),
                         np.arange(ymin, ymax, h))
    Z = knn2.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

    ax.contourf(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
                cmap=plt.colormaps["Pastel1"].resampled(4), alpha=0.55)
    ax.contour(xx, yy, Z, levels=[-0.5, 0.5, 1.5, 2.5, 3.5],
               colors="white", linewidths=0.6, alpha=0.7)

    for c, color, marcador, nombre in zip(range(4), PALETA, MARCADORES, NOMBRES_CLASES):
        ax.scatter(X_tr2[y_tr == c, 0], X_tr2[y_tr == c, 1],
                   c=color, marker=marcador, s=70, edgecolors="white",
                   linewidths=0.8, label=nombre, zorder=4)
        ax.scatter(X_te2[y_te == c, 0], X_te2[y_te == c, 1],
                   c=color, marker=marcador, s=160, edgecolors="black",
                   linewidths=1.8, zorder=5)

    ax.set_xlabel(f"{FEATURES[i]} (escalado)", fontsize=10)
    ax.set_ylabel(f"{FEATURES[j]} (escalado)", fontsize=10)
    ax.set_title(f"{FEATURES[i]}  vs  {FEATURES[j]}", fontsize=11)

handles = ([plt.scatter([], [], c=PALETA[c], marker=MARCADORES[c],
                        s=70, label=NOMBRES_CLASES[c]) for c in range(4)] +
           [plt.scatter([], [], c="gray", marker="o", s=70,
                        edgecolors="white", label="Train"),
            plt.scatter([], [], c="gray", marker="o", s=160,
                        edgecolors="black", linewidths=2, label="Prueba")])
fig.legend(handles=handles, loc="lower center", ncol=6,
           fontsize=9, bbox_to_anchor=(0.5, -0.08), frameon=True)
fig.suptitle(
    f"Fronteras de decisión KNN (k={k_gs}, {dist_gs}, weights={w_gs})",
    fontsize=13, y=1.02
)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_05_fronteras.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_05_fronteras.png")

# ============================================================
# 10. IMPORTANCIA DE FEATURES (Permutation Importance)
# ============================================================
perm     = permutation_importance(knn, X_va_s, y_va,
                                  n_repeats=50, random_state=42,
                                  scoring="accuracy")
imp_mean = perm.importances_mean
imp_std  = perm.importances_std
orden    = np.argsort(imp_mean)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.barh(
    [FEATURES[i] for i in orden], imp_mean[orden],
    xerr=imp_std[orden],
    color=[PALETA[k % 4] for k in range(len(FEATURES))],
    edgecolor="white", height=0.5, capsize=4
)
ax.axvline(0, color="gray", linewidth=0.8, linestyle="--")
for bar, val in zip(bars, imp_mean[orden]):
    ax.text(max(val, 0) + 0.002, bar.get_y() + bar.get_height() / 2,
            f"{val:+.3f}", va="center", fontsize=9)
ax.set_xlabel("Reducción media de accuracy al permutar", fontsize=11)
ax.set_title("Importancia de features — Permutation Importance\n"
             "(validación, 50 repeticiones)", fontsize=11)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_06_importancia.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_06_importancia.png")

# ============================================================
# 11. DISTRIBUCIÓN DE FEATURES POR CLASE
# ============================================================
fig, axes = plt.subplots(1, len(FEATURES), figsize=(5 * len(FEATURES), 4))
for ax, feat in zip(axes, FEATURES):
    for c, (color, nombre) in enumerate(zip(PALETA, NOMBRES_CLASES)):
        vals = df[df["Clase"] == c][feat].values
        ax.scatter([nombre] * len(vals), vals, color=color, s=60,
                   edgecolors="white", linewidths=0.8, zorder=3, alpha=0.85)
        if len(vals):
            ax.plot([nombre], [vals.mean()], marker="_", markersize=25,
                    color="black", linewidth=2.5, zorder=4)
    ax.set_xlabel("Clase", fontsize=10)
    ax.set_ylabel(feat,    fontsize=10)
    ax.set_title(f"Distribución de {feat}", fontsize=11)
    ax.tick_params(axis="x", labelsize=9)
fig.suptitle("Distribución de features por clase  (─ = media)",
             fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_07_distribucion_features.png",
            dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_07_distribucion_features.png")

# ============================================================
# 12. GRÁFICO EXTRA: distribución por grupo de recolección
# ============================================================
grupos_orden = ["C", "X", "J", "D", "V", "SG", "S"]
fig, ax = plt.subplots(figsize=(10, 4))
bottom = np.zeros(len(grupos_orden))
for c, (color, nombre) in enumerate(zip(PALETA, NOMBRES_CLASES)):
    counts = [int((df[df["grupo"] == g]["Clase"] == c).sum())
              for g in grupos_orden]
    ax.bar(grupos_orden, counts, bottom=bottom, color=color,
           label=nombre, edgecolor="white", width=0.6)
    bottom += np.array(counts)
ax.set_xlabel("Grupo de recolección", fontsize=11)
ax.set_ylabel("Número de hojas",       fontsize=11)
ax.set_title("Composición por grupo y clase", fontsize=12)
ax.legend(fontsize=10, bbox_to_anchor=(1.01, 1), loc="upper left")
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}/v2_08_grupos.png", dpi=150, bbox_inches="tight")
plt.close()
print("  → v2_08_grupos.png")

# ============================================================
# 13. RESUMEN FINAL
# ============================================================
f1s_te        = f1_score(y_te, y_pred_te, average=None,
                         zero_division=0, labels=[0, 1, 2, 3])
clase_dificil = NOMBRES_CLASES[np.argmin(f1s_te)]
feat_clave    = FEATURES[np.argmax(imp_mean)]

from sklearn.metrics import classification_report as cr_fn

def rep_dict(y_true, y_pred):
    clases  = sorted(set(y_true) | set(y_pred))
    nombres = [NOMBRES_CLASES[c] for c in clases]
    return cr_fn(y_true, y_pred, labels=clases, target_names=nombres,
                 zero_division=0, output_dict=True)

rep_va = rep_dict(y_va, y_pred_va)
rep_te = rep_dict(y_te, y_pred_te)

print("\n" + "=" * 65)
print(f"RESUMEN FINAL — KNN v3 | n={len(df)} muestras")
print("=" * 65)
print(f"  k óptimo (LOO-CV)    : {k_optimo_loo}")
print(f"  Mejor config GS      : k={k_gs}, metric={dist_gs}, weights={w_gs}")
print(f"  F1-macro (GS-CV)     : {grid_search.best_score_:.3f}")
print(f"  Accuracy validación  : {acc_va:.3f}")
print(f"  Accuracy prueba      : {acc_te:.3f}")
print(f"  F1-macro prueba      : {rep_te['macro avg']['f1-score']:.3f}")
print(f"  Feature más útil     : {feat_clave}  (Δacc={max(imp_mean):.3f})")
print(f"  Clase más difícil    : {clase_dificil}  (F1={min(f1s_te):.3f})")
print()
print("  F1 por clase (prueba):")
for i, n in enumerate(NOMBRES_CLASES):
    print(f"    {n:12s}: {f1s_te[i]:.3f}")
print()
print("  LOO scores:")
for k, s in zip(k_range, loo_scores):
    print(f"    k={k:2d} → {s:.3f}")
print("=" * 65)

# ── Guardar resultados en JSON ──────────────────────────────
import json

def cm_txt(y_true, y_pred):
    clases = sorted(set(y_true) | set(y_pred))
    ets    = [NOMBRES_CLASES[c] for c in clases]
    cm     = confusion_matrix(y_true, y_pred, labels=clases)
    w      = max(len(e) for e in ets) + 2
    lineas = [" " * w + "  " + "  ".join(f"{e:>{w}}" for e in ets)]
    for i2, fila in enumerate(cm):
        lineas.append(f"{ets[i2]:>{w}}  " + "  ".join(f"{v:>{w}}" for v in fila))
    return "\n".join(lineas)

results = {
    "n_total": int(len(df)),
    "n_train": int(len(X_tr)),
    "n_val":   int(len(X_va)),
    "n_test":  int(len(X_te)),
    "grupos":  df["grupo"].value_counts().sort_index().to_dict(),
    "clases_n": {NOMBRES_CLASES[i]: int((y_all == i).sum()) for i in range(4)},
    "k_loo":   int(k_optimo_loo),
    "k_gs":    int(k_gs),
    "dist_gs": dist_gs,
    "w_gs":    w_gs,
    "f1_macro_gs":  round(float(grid_search.best_score_), 3),
    "acc_va":       round(float(acc_va), 3),
    "acc_te":       round(float(acc_te), 3),
    "f1_macro_va":  round(float(rep_va["macro avg"]["f1-score"]), 3),
    "f1_macro_te":  round(float(rep_te["macro avg"]["f1-score"]), 3),
    "feat_clave":   feat_clave,
    "clase_dificil": clase_dificil,
    "f1_min":       round(float(min(f1s_te)), 3),
    "f1_por_clase": {NOMBRES_CLASES[i]: round(float(f1s_te[i]), 3) for i in range(4)},
    "imp_mean": {FEATURES[i]: round(float(imp_mean[i]), 4) for i in range(len(FEATURES))},
    "loo_scores": [round(s, 3) for s in loo_scores],
    "k_range":  list(k_range),
    "rep_va_por_clase": {
        k: {m: round(v, 3) for m, v in vals.items()}
        for k, vals in rep_va.items()
        if k not in ("accuracy", "macro avg", "weighted avg")
    },
    "rep_te_por_clase": {
        k: {m: round(v, 3) for m, v in vals.items()}
        for k, vals in rep_te.items()
        if k not in ("accuracy", "macro avg", "weighted avg")
    },
    "rep_va_macro":    {m: round(rep_va["macro avg"][m], 3)
                        for m in ["precision", "recall", "f1-score"]},
    "rep_te_macro":    {m: round(rep_te["macro avg"][m], 3)
                        for m in ["precision", "recall", "f1-score"]},
    "rep_va_weighted": {m: round(rep_va["weighted avg"][m], 3)
                        for m in ["precision", "recall", "f1-score"]},
    "rep_te_weighted": {m: round(rep_te["weighted avg"][m], 3)
                        for m in ["precision", "recall", "f1-score"]},
    "cm_val_text":  cm_txt(y_va, y_pred_va),
    "cm_test_text": cm_txt(y_te, y_pred_te),
}

with open(f"{OUTPUT_DIR}/knn_results_v3.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2, ensure_ascii=False)

print(f"\nArchivos generados en '{OUTPUT_DIR}/':")
for nombre in ["v2_01_seleccion_k", "v2_02_matrices_confusion",
               "v2_03_roc_curves",  "v2_04_pr_curves",
               "v2_05_fronteras",   "v2_06_importancia",
               "v2_07_distribucion_features", "v2_08_grupos",
               "knn_results_v3"]:
    ext = ".png" if not nombre.startswith("knn") else ".json"
    print(f"  {nombre}{ext}")

DATASET — n=114 muestras | 7 grupos

Distribución por grupo:
  C   : 50 muestras
  D   : 8 muestras
  J   : 12 muestras
  S   : 6 muestras
  SG  : 7 muestras
  V   : 6 muestras
  X   : 25 muestras

Distribución por clase:
  0 Alargada    : ██████████████████████████████████████████████████████ (54)
  1 Ovalada     : ███████████████████████ (23)
  2 Estrellada  : ██████████████████████████████ (30)
  3 Helecho     : ███████ (7)

  Total: 114 muestras

  Entrenamiento: 68 | Validación: 23 | Prueba: 23

  K óptimo LOO-CV : 5  (acc = 0.750)
  GridSearch best : k=4, metric=manhattan, weights=uniform, F1-macro=0.695
  → v2_01_seleccion_k.png
  → v2_02_matrices_confusion.png

────────────────────────────────────────────────────────────
  VALIDACIÓN  (n=23)
────────────────────────────────────────────────────────────
              precision    recall  f1-score   support

    Alargada       0.71      0.91      0.80        11
     Ovalada       0.50      0.20      0.29         5
  Estrellada    